# Lista 5 — Zadanie 5: Eksploracja parametrów LLM (30 pkt)

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas langchain-core langchain-huggingface pydantic accelerate bitsandbytes

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import json
import re

import torch
import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions

## Krok 1: Konfiguracja

In [ ]:
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SAMPLE_SIZE = 80

examples = load_polemo_test()
if SAMPLE_SIZE is not None:
    examples = examples[:SAMPLE_SIZE]

sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]
print(f"Próbek: {len(sentences)} | GPU: {torch.cuda.is_available()}")

## Krok 2: Definicje promptów

In [ ]:
PROMPT_SIMPLE = """Classify the text sentiment into one of three classes: positive, negative, neutral.
Reply with only one word.

Text: {text}
Class:"""

PROMPT_DETAILED = """Jesteś klasyfikatorem wydźwięku polskich recenzji.
Przypisz tekst do dokładnie jednej klasy:
- positive — opinia jednoznacznie pozytywna
- negative — opinia jednoznacznie negatywna
- neutral — opinia opisowa, bez wyraźnych emocji

Odpowiedz jednym słowem (positive, negative lub neutral).

Tekst: {text}
Klasa:"""

PROMPT_JSON = """Jesteś klasyfikatorem wydźwięku. Zwróć odpowiedź jako JSON.

Tekst: {text}

{format_instructions}"""


class SentimentResult(BaseModel):
    sentiment: str = Field(description="One of: positive, negative, neutral")


json_parser = JsonOutputParser(pydantic_object=SentimentResult)
json_format_instructions = json_parser.get_format_instructions()

## Krok 3: Funkcje do eksperymentów

In [ ]:
import gc
import time

_loaded_model = None
_loaded_tokenizer = None
_loaded_quantization = None


def reset_llm_cache():
    global _loaded_model, _loaded_tokenizer, _loaded_quantization
    _loaded_model = None
    _loaded_tokenizer = None
    _loaded_quantization = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_model(quantization="float16"):
    global _loaded_model, _loaded_tokenizer, _loaded_quantization

    if _loaded_model is not None and _loaded_quantization == quantization:
        return _loaded_model, _loaded_tokenizer

    reset_llm_cache()
    _loaded_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

    if quantization == "4bit":
        from transformers import BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        _loaded_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        _loaded_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )

    _loaded_quantization = quantization
    return _loaded_model, _loaded_tokenizer


def get_llm(temperature=0.1, max_new_tokens=15, quantization="float16"):
    model, tokenizer = load_model(quantization=quantization)

    hf_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        temperature=temperature,
        do_sample=temperature > 0,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )
    return HuggingFacePipeline(pipeline=hf_pipe)


def extract_label_text(answer: str) -> str:
    if not answer:
        return ""

    for marker in ("Class:", "Klasa:"):
        if marker in answer:
            answer = answer.rsplit(marker, 1)[-1]

    first_line = answer.strip().splitlines()[0].strip()
    return first_line.split()[0] if first_line else ""


def parse_text_answer(answer: str) -> str:
    mapped = map_text_to_class(extract_label_text(answer))
    return mapped if mapped else "neutral"


def parse_json_answer(answer: str) -> str:
    json_match = re.search(
        r'\{[^{}]*"sentiment"\s*:\s*"[^"]+"[^{}]*\}',
        answer,
        re.IGNORECASE | re.DOTALL,
    )
    json_text = json_match.group() if json_match else answer

    try:
        parsed = json_parser.parse(json_text)
        sentiment = parsed.get("sentiment", "")
        return parse_text_answer(sentiment)
    except Exception:
        match = re.search(r"\{[^}]+\}", answer)
        if match:
            try:
                data = json.loads(match.group())
                return parse_text_answer(str(data.get("sentiment", "")))
            except json.JSONDecodeError:
                pass
        return parse_text_answer(answer)


def run_llm_experiment(
    name,
    prompt_template,
    temperature,
    parse_fn,
    use_json_format=False,
    quantization="float16",
):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    llm = get_llm(temperature=temperature, quantization=quantization)

    prompt = PromptTemplate.from_template(prompt_template)
    chain = prompt | llm

    y_pred = []
    for sentence in tqdm(sentences, desc=name):
        if use_json_format:
            answer = chain.invoke({
                "text": sentence,
                "format_instructions": json_format_instructions,
            })
        else:
            answer = chain.invoke({"text": sentence})
        y_pred.append(parse_fn(answer))

    elapsed = time.time() - start
    metrics = evaluate_predictions(y_true, y_pred)

    vram_gb = None
    if torch.cuda.is_available():
        vram_gb = torch.cuda.max_memory_allocated() / 1e9

    return {
        "eksperyment": name,
        "temperature": temperature,
        "kwantyzacja": quantization,
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
        "czas_s": round(elapsed, 1),
        "vram_gb": round(vram_gb, 2) if vram_gb is not None else None,
    }

## Eksperyment A: Wpływ temperatury (prosty prompt)

In [ ]:
TEMPERATURES = [0.0, 0.1, 0.7]
temp_results = []

for temp in TEMPERATURES:
    result = run_llm_experiment(
        name=f"temp={temp}",
        prompt_template=PROMPT_SIMPLE,
        temperature=temp,
        parse_fn=parse_text_answer,
    )
    temp_results.append(result)
    print(f"temp={temp}: accuracy={result['accuracy']:.4f}, f1_macro={result['f1_macro']:.4f}")

## Eksperyment B: Wpływ promptu (temperatura=0.1)

In [ ]:
prompt_results = []

for prompt_name, prompt_text in [("prosty", PROMPT_SIMPLE), ("szczegółowy", PROMPT_DETAILED)]:
    result = run_llm_experiment(
        name=f"prompt={prompt_name}",
        prompt_template=prompt_text,
        temperature=0.1,
        parse_fn=parse_text_answer,
    )
    result["prompt"] = prompt_name
    prompt_results.append(result)
    print(f"prompt={prompt_name}: accuracy={result['accuracy']:.4f}, f1_macro={result['f1_macro']:.4f}")

## Eksperyment C: Parsowanie JSON (JsonOutputParser)

In [ ]:
json_result = run_llm_experiment(
    name="parsowanie=JSON",
    prompt_template=PROMPT_JSON,
    temperature=0.1,
    parse_fn=parse_json_answer,
    use_json_format=True,
)
json_result["prompt"] = "JSON"
print(f"JSON parser: accuracy={json_result['accuracy']:.4f}, f1_macro={json_result['f1_macro']:.4f}")

## Eksperyment D: Kwantyzacja (float16 vs 4-bit)

In [ ]:
QUANT_MODES = [
    ("float16", "Bez kwantyzacji (float16)"),
    ("4bit", "Kwantyzacja 4-bit (NF4)"),
]

quant_results = []

for mode, label in QUANT_MODES:
    if mode == "4bit" and not torch.cuda.is_available():
        print("Pominięto 4-bit — brak GPU")
        continue

    reset_llm_cache()

    try:
        result = run_llm_experiment(
            name=f"quant={mode}",
            prompt_template=PROMPT_SIMPLE,
            temperature=0.1,
            parse_fn=parse_text_answer,
            quantization=mode,
        )
        quant_results.append(result)
        print(
            f"{label}: accuracy={result['accuracy']:.4f}, "
            f"f1_macro={result['f1_macro']:.4f}, "
            f"czas={result['czas_s']}s, VRAM={result['vram_gb']} GB"
        )
    except Exception as e:
        print(f"Błąd dla {label}: {e}")
    finally:
        reset_llm_cache()

pd.DataFrame(quant_results)

## Krok 4: Tabela porównawcza wszystkich eksperymentów

In [ ]:
all_results = temp_results + prompt_results + [json_result] + quant_results
comparison_df = pd.DataFrame(all_results)
comparison_df.sort_values("f1_macro", ascending=False)